In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, Matern

import matplotlib.pyplot as plt
import time

from bayes_opt import BayesianOptimization


# Example mortality data (replace this with your actual dataset)
file_path = "HMD_20250418A.csv"
df = pd.read_csv(file_path)

# have a brief look at data
#print(df.head())

In [2]:
# "countries" to exclude initially, for various reasons
exclude_countries = [
    "England & Wales Civilian", "FranceCivilian", "GDR", "New Zealand Maori", 
    "New Zealand non-Maori", "UK", "WestGermany"
]
# Remove excluded countries and exclude ages below 16
filtered_df = df[~df["Country"].isin(exclude_countries) ]

# Get min and max year for each country
year_stats = filtered_df.groupby("Country")["Year"].agg(["min", "max"]).reset_index()

print(year_stats)

# Get 10 highest minimum years
highest_mins = year_stats.nlargest(10, "min")

# Get 10 lowest maximum years
lowest_maxes = year_stats.nsmallest(10, "max")
# Get the 10 highhest maximum years
highest_maxes = year_stats.nlargest(10, "max")

print("10 highest Minimum Years:")
print(highest_mins)

print("10 lowest and 10 highest Maximum Years:")
print(lowest_maxes, highest_maxes)

                  Country   min   max
0               Australia  1921  2021
1                 Austria  1947  2023
2                 Belarus  1959  2018
3                 Belgium  1841  2023
4                Bulgaria  1947  2021
5                  Canada  1921  2022
6                   Chile  1992  2020
7                 Croatia  2001  2020
8                 Czechia  1950  2021
9                 Denmark  1835  2024
10  England & Wales Total  1841  2022
11                Estonia  1959  2019
12                Finland  1878  2023
13              FranceAll  1816  2022
14                Germany  1990  2020
15                 Greece  1981  2019
16               HongKong  1986  2023
17                Hungary  1950  2020
18                Iceland  1838  2022
19                Ireland  1950  2022
20                 Israel  1983  2016
21                  Italy  1872  2021
22                  Japan  1947  2023
23                 Latvia  1959  2019
24              Lithuania  1959  2020
25          

In [3]:
# Filter for countries where the minimum year is <= 1990 and the maximum year is >= 2019
core_countries = year_stats[(year_stats["min"] <= 1990) & (year_stats["max"] >= 2019)]["Country"]
print("Core Countries: ", core_countries)

#filter for 1990 to 2019 years only
core_countries_years = filtered_df[(filtered_df["Year"] >= 1990) & (filtered_df["Year"] <= 2019) & (filtered_df["Country"].isin(core_countries))]
print(core_countries_years)

# Reshape the dataframe to have separate rows for Male and Female
df_melted = core_countries_years.melt(id_vars=["Country", "Year", "Age"], value_vars=["F", "M"], 
                     var_name="Sex", value_name="MortalityRate")

# Convert gender labels to meaningful names
df_melted["Sex"] = df_melted["Sex"].replace({"F": "Female", "M": "Male"})

print(df_melted)

# Write the dataframe to an Excel file
df_melted.to_csv("filtered_mortality_data.csv", index=False)

data_initial = df_melted


Core Countries:  0                 Australia
1                   Austria
3                   Belgium
4                  Bulgaria
5                    Canada
8                   Czechia
9                   Denmark
10    England & Wales Total
11                  Estonia
12                  Finland
13                FranceAll
14                  Germany
15                   Greece
16                 HongKong
17                  Hungary
18                  Iceland
19                  Ireland
21                    Italy
22                    Japan
23                   Latvia
24                Lithuania
25               Luxembourg
26              Netherlands
27              New Zealand
28         Northern Ireland
29                   Norway
30                   Poland
31                 Portugal
33                 Scotland
34                 Slovakia
35                 Slovenia
37                    Spain
38                   Sweden
39              Switzerland
40                   Taiwan
41 

In [4]:
# Make a copy to prevent unintended slice modifications
data_initial = data_initial.copy()

# Replace "110+" with 110
data_initial["Age"] = data_initial["Age"].replace("110+", 110)

# Convert Age column to numeric
data_initial["Age"] = pd.to_numeric(data_initial["Age"], errors="coerce")

# Ensure MortalityRate is fully numeric by filtering non-convertible values
data_initial.loc[:, "MortalityRate"] = pd.to_numeric(data_initial["MortalityRate"], errors="coerce")

# NaN replaced with last numeric value above
data_initial["MortalityRate"] = data_initial["MortalityRate"].bfill()

# Replace zero values with a small positive number to prevent log(0)
data_initial.loc[data_initial["MortalityRate"] == 0, "MortalityRate"] = 0.00001

# Convert the column to a numeric type again and explicitly enforce float
data_initial["MortalityRate"] = pd.to_numeric(data_initial["MortalityRate"], errors="coerce").astype(float)

# Restrict dataset to ages 16 and above
data_initial = data_initial[data_initial["Age"] >= 16]


# Now, apply the log transformation safely
data_initial["MortalityRate"] = np.log(data_initial["MortalityRate"])

# Debugging: Check if any non-numeric values still exist
non_numeric_rows = data_initial.loc[~data_initial["MortalityRate"].apply(lambda x: isinstance(x, (int, float))), ["Age", "MortalityRate"]]

# Group by MortalityRate and aggregate ages into a list
summary = non_numeric_rows.groupby("MortalityRate")["Age"].apply(list).reset_index()

# Display the summarized data
print(summary)

print(data_initial)


Empty DataFrame
Columns: [MortalityRate, Age]
Index: []
          Country  Year  Age     Sex  MortalityRate
16      Australia  1990   16  Female      -8.069307
17      Australia  1990   17  Female      -7.932188
18      Australia  1990   18  Female      -7.411936
19      Australia  1990   19  Female      -7.768138
20      Australia  1990   20  Female      -7.489361
...           ...   ...  ...     ...            ...
239755        USA  2019  106    Male      -0.549322
239756        USA  2019  107    Male      -0.439904
239757        USA  2019  108    Male      -0.199343
239758        USA  2019  109    Male      -0.798257
239759        USA  2019  110    Male      -0.848011

[205200 rows x 5 columns]


C:\Users\steph\AppData\Local\Temp\ipykernel_27716\337972328.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_initial["MortalityRate"] = data_initial["MortalityRate"].bfill()


In [ ]:
# Function to optimize Gaussian Process Regression
def optimize_gpr(data, country, sex, cutoff_year_adjusted):
    prev_best_params = {"length_scale": 10, "alpha": 1e-2}

    optimizer = BayesianOptimization(
        f=lambda length_scale, alpha: -gpr_evaluate(length_scale, alpha, data, country, sex, cutoff_year_adjusted)[0]["Train_MSE"],
        pbounds={"length_scale": (2, 50),  # Adjusted lower bound to prevent premature convergence
                 "alpha": (1e-6, 0.05)},  # Refined alpha range for better tuning
        random_state=42
    )

    optimizer.maximize(init_points=5, n_iter=15)  # Run optimization

    return optimizer.max["params"]

# Adjust year values relative to 1990
data_initial["Year_Adjusted"] = data_initial["Year"] - 1990

# Function to optimize Gaussian Process Regression
def optimize_gpr(data, country, sex, cutoff_year_adjusted):
    prev_best_params = {"length_scale": 10, "alpha": 1e-2}
    start_time = time.time()

    optimizer = BayesianOptimization(
        f=lambda length_scale, alpha: -gpr_evaluate(length_scale, alpha, data, country, sex, cutoff_year_adjusted)[0]["Train_MSE"],
        pbounds={"length_scale": (2, 50),  # Adjusted lower bound to prevent premature convergence
                 "alpha": (1e-6, 0.05)},  # Refined alpha range for better tuning
        random_state=42
    )

    optimizer.maximize(init_points=5, n_iter=15)  # Run optimization
    init_time = time.time() - start_time
    iter_time = init_time / max(len(optimizer.res), 1)  # Prevent division by zero

    return optimizer.max["params"], init_time, iter_time

# Function to train and evaluate GPR model
def gpr_evaluate(length_scale, alpha, data, country, sex, cutoff_year_adjusted):
    subset = data[(data["Country"] == country) & (data["Sex"] == sex)].copy()
    train_data = subset[subset["Year_Adjusted"] <= cutoff_year_adjusted]

    X_train = train_data[["Year_Adjusted", "Age"]].values
    y_train = train_data["MortalityRate"].values

    # Standardize X_train before fitting
    scaler_X = StandardScaler()
    X_train_std = scaler_X.fit_transform(X_train)

    # Use Matern kernel with optimized length scale and refined parameters
    kernel = C(1.0) * Matern(length_scale=length_scale, length_scale_bounds=(2, 50), nu=1.5)

    gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=alpha,
    n_restarts_optimizer=20  # Increase restarts to improve optimization stability
    )

    gpr.fit(X_train_std, y_train)  # Fit using standardized X_train

    # Define future years dynamically
    future_years = subset[subset["Year_Adjusted"] > cutoff_year_adjusted]["Year_Adjusted"].unique()
    test_results = []
    ages = list(range(16, 111))  # Ages 16 to 110

    for fy in future_years:
        X_future = subset[subset["Year_Adjusted"] == fy][["Year_Adjusted", "Age"]].values
        y_future = subset[subset["Year_Adjusted"] == fy]["MortalityRate"].values  # Actual values

        # If no data for that year, create synthetic entries for all ages
        if len(X_future) == 0:
            X_future = np.array([[fy, age] for age in ages])  # Ensure all ages exist
            y_future = np.full((len(ages),), np.nan)  # Assign NaNs for missing actual data

        # Standardize X_future before predictions
        X_future_std = scaler_X.transform(X_future)

        y_pred, _ = gpr.predict(X_future_std, return_std=True)

        for age, pred_value, actual_value in zip(ages, y_pred, y_future):
            test_results.append({
                "Year_Adjusted": fy,
                "Age": age,
                "Predicted_MortalityRate": pred_value,
                "Actual_MortalityRate": actual_value
            })

        # Compute fit metrics only for years where actual data exists
        if not np.isnan(y_future).all():
            mse = mean_squared_error(y_future, y_pred)
            mae = mean_absolute_error(y_future, y_pred)
            r2 = r2_score(y_future, y_pred)
        else:
            mse, mae, r2 = np.nan, np.nan, np.nan  # Assign NaNs if no actual values exist

        test_results.append({"Year_Adjusted": fy, "MSE": mse, "MAE": mae, "R2": r2})

    train_mse = mean_squared_error(y_train, gpr.predict(X_train_std))
    train_mae = mean_absolute_error(y_train, gpr.predict(X_train_std))
    train_r2 = r2_score(y_train, gpr.predict(X_train_std))

    return {"Train_MSE": train_mse, "Train_MAE": train_mae, "Train_R2": train_r2}, test_results, future_years

# Running optimization for each country-sex set across cutoff years
cutoff_years = [2000, 2005, 2010, 2015, 2019]
best_params_tracker = {}

#for country in data_initial["Country"].unique():
for country in ["England & Wales Total"]:
    for sex in ["Male", "Female"]:
        subset = data_initial[(data_initial["Country"] == country) & (data_initial["Sex"] == sex)].copy()
        X = subset[["Year_Adjusted", "Age"]].values
        Y = subset["MortalityRate"].values.reshape(-1, 1)

        # Standardize Y only
        scaler_Y = StandardScaler()
        scaler_Y.fit(Y)
        Y_std = scaler_Y.transform(Y)

        cutoff_years_adjusted = [year - 1990 for year in cutoff_years]

        for cutoff_year_adjusted in cutoff_years_adjusted:
            best_params, init_time, iter_time = optimize_gpr(data_initial, country, sex, cutoff_year_adjusted)
            train_metrics, test_results, future_years = gpr_evaluate(
                best_params["length_scale"], best_params["alpha"], data_initial, country, sex, cutoff_year_adjusted
            )

            print("\n" + "="*50)
            print(f"Processing: Country: {country}, Sex: {sex}, Cutoff Year: {cutoff_year_adjusted + 1990}")
            print(f"Optimized Hyperparameters: {best_params}")
            print(f"Init Time: {init_time:.4f} sec | Iteration Time: {iter_time:.4f} sec")
            print("\nTraining Metrics:")
            print(f"MSE: {train_metrics['Train_MSE']:.4f}, MAE: {train_metrics['Train_MAE']:.4f}, R2: {train_metrics['Train_R2']:.4f}")

            # Extract actual mortality rates for ALL remaining years
            actual_future_rates = subset[subset["Year_Adjusted"] > cutoff_year_adjusted][["Year_Adjusted", "Age", "MortalityRate"]]
            
            # Convert test results into a DataFrame for better comparison
            pred_df = pd.DataFrame(test_results)
            
            # Ensure predictions include all ages in every future year
            ages = list(range(16, 111))
            expanded_pred = [{"Year_Adjusted": fy, "Age": age} for fy in future_years for age in ages]
            expanded_pred_df = pd.DataFrame(expanded_pred)

            # Merge expanded Age-Year grid with predicted results
            pred_df = pd.merge(expanded_pred_df, pred_df, on=["Year_Adjusted", "Age"], how="left")

            # Merge predictions with actual mortality rates
            comparison_df = pd.merge(pred_df, actual_future_rates, on=["Year_Adjusted", "Age"], how="left")

            print("\nPredicted vs Actual Mortality Rates:")
            print(comparison_df)

            # Save results to CSV
            filename = f"mortality_results_{country}_{sex}_{cutoff_year_adjusted+2000}.csv"

            # Concatenate both datasets into one file
            final_df = pd.concat([train_metrics_df, comparison_df], axis=0)

            final_df.to_csv(filename, index=False)
            print(f"Results saved to {filename}")



|   iter    |  target   |   alpha   | length... |
-------------------------------------------------


C:\Users\steph\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 2. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


| 1         | -0.001084 | 0.01873   | 47.63     |


C:\Users\steph\anaconda3\Lib\site-packages\sklearn\gaussian_process\_gpr.py:659: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
